# S6E9 Last-Day Blends

blend 1: .94637
blend 2: .94641
blend 3: .94640
blend 4: .94640
blend 5: .94640
blend 6: .94638
blend 7: .94639
blend 8: .94641

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

RNG = np.random.default_rng(42)
OUT_DIR = Path("blends")
OUT_DIR.mkdir(exist_ok=True)

train = pd.read_csv("playground-series-s6e9/train.csv", usecols=["id", "Will_Buy_EV"])
test = pd.read_csv("playground-series-s6e9/test.csv", usecols=["id"])
y = (train["Will_Buy_EV"] == "Yes").astype(np.int8).to_numpy()
ids_tr = train["id"].to_numpy()
ids_te = test["id"].to_numpy()
n_tr, n_te = len(ids_tr), len(ids_te)
print(f"train={n_tr:,}  test={n_te:,}  pos={y.mean():.4f}")

train=668,665  test=286,571  pos=0.1746


## Load pool

In [2]:
def load_sub(path, id_index, col="Will_Buy_EV"):
    df = pd.read_csv(path)
    if col not in df.columns:
        col = [c for c in df.columns if c != "id"][0]
    return df.set_index("id").reindex(id_index)[col].to_numpy(dtype=np.float64)


def rank01(a):
    return rankdata(a.astype(np.float64), method="average") / (len(a) + 1.0)


def save_blend(name, pred, note=""):
    pred = np.asarray(pred, dtype=np.float64)
    assert pred.shape == (n_te,), (name, pred.shape)
    assert np.isfinite(pred).all(), name
    out = pd.DataFrame({"id": ids_te, "Will_Buy_EV": pred})
    path = OUT_DIR / f"{name}.csv"
    out.to_csv(path, index=False)
    print(
        f"saved {path.name:40s} mean={pred.mean():.6f} "
        f"std={pred.std():.6f} nunique={pd.Series(pred).nunique():,}  {note}"
    )
    return path


# --- own LB submissions (competitive band) ---
own_files = {
    "s94639": ("score94639.csv", 0.94639),
    "s94637": ("score94637.csv", 0.94637),
    "s94637b": ("score94637(1).csv", 0.94637),
    "s94636": ("score94636.csv", 0.94636),
    "s94634": ("score94634.csv", 0.94634),
    "s94629": ("score94629.csv", 0.94629),
    "s94613": ("score94613.csv", 0.94613),
    "s94595": ("score94595.csv", 0.94595),
}
own_te = {k: load_sub(p, ids_te) for k, (p, _) in own_files.items()}
own_lb = {k: lb for k, (_, lb) in own_files.items()}

# --- externals with OOF + test ---
ext = {}

def add_ext(key, oof, te, lb):
    ext[key] = {"oof": np.asarray(oof, dtype=np.float64), "te": np.asarray(te, dtype=np.float64), "lb": lb}

fv_oof = pd.read_csv("external/mega_four_views/oof.csv").set_index("id").reindex(ids_tr)
fv_te = pd.read_csv("external/mega_four_views/test.csv").set_index("id").reindex(ids_te)
add_ext("fvE", fv_oof["ensemble_ABCD"], fv_te["ensemble_ABCD"], 0.94645)
add_ext("fvA", fv_oof["A_lgbm_triple_te_3seed"], fv_te["A_lgbm_triple_te_3seed"], 0.9463)
add_ext("fvB", fv_oof["B_xgb_same_features"], fv_te["B_xgb_same_features"], 0.9463)
add_ext("fvC", fv_oof["C_lgbm_windows_lift_nodigit"], fv_te["C_lgbm_windows_lift_nodigit"], 0.9462)
add_ext("fvD", fv_oof["D_lgbm_alt_ladder_noexact"], fv_te["D_lgbm_alt_ladder_noexact"], 0.9461)

add_ext("mega", np.load("external/mega_lgb94633/oof_hybrid.npy"), np.load("external/mega_lgb94633/test_hybrid.npy"), 0.94633)
add_ext("herm", np.load("external/herm_xgb94627/oof.npy").ravel(), np.load("external/herm_xgb94627/test.npy").ravel(), 0.94627)
add_ext(
    "naji",
    load_sub("external/naji_lgb94638/oof.csv", ids_tr, "OOF_Pred"),
    load_sub("external/naji_lgb94638/test.csv", ids_te),
    0.94638,
)
add_ext(
    "nxgb",
    load_sub("external/naji_xgb94639/oof.csv", ids_tr, "OOF_Pred"),
    load_sub("external/naji_xgb94639/test.csv", ids_te),
    0.94639,
)
add_ext(
    "ncb",
    load_sub("external/naji_cb94624/oof.csv", ids_tr, "OOF_Pred"),
    load_sub("external/naji_cb94624/test.csv", ids_te),
    0.94624,
)
add_ext(
    "rtab",
    load_sub("external/realmlp_tab94620/oof.csv", ids_tr),
    load_sub("external/realmlp_tab94620/test.csv", ids_te),
    0.94620,
)
add_ext(
    "rmlp",
    load_sub("external/realmlp_pt94621/oof.csv", ids_tr),
    load_sub("external/realmlp_pt94621/test.csv", ids_te),
    0.94621,
)
add_ext(
    "pub",
    np.load("external/pub_lgb94607/oof_predictions_lgb_42_5.npy"),
    np.load("external/pub_lgb94607/test_predictions_lgb_42_5.npy"),
    0.94607,
)

for k, v in ext.items():
    print(f"ext {k:5s} oof={roc_auc_score(y, v['oof']):.6f}  lb~{v['lb']:.5f}")

ANCHOR = "s94639"
print("anchor", ANCHOR, "lb", own_lb[ANCHOR], "sp_fvE", spearmanr(own_te[ANCHOR], ext["fvE"]["te"]).correlation)

ext fvE   oof=0.946339  lb~0.94645
ext fvA   oof=0.946281  lb~0.94630
ext fvB   oof=0.946258  lb~0.94630
ext fvC   oof=0.946223  lb~0.94620
ext fvD   oof=0.946077  lb~0.94610
ext mega  oof=0.946264  lb~0.94633
ext herm  oof=0.946057  lb~0.94627
ext naji  oof=0.946068  lb~0.94638
ext nxgb  oof=0.946142  lb~0.94639
ext ncb   oof=0.946099  lb~0.94624
ext rtab  oof=0.946065  lb~0.94620
ext rmlp  oof=0.946010  lb~0.94621
ext pub   oof=0.945832  lb~0.94607
anchor s94639 lb 0.94639 sp_fvE 0.9999185822116546


## Blend helpers

In [3]:
def weighted_rank(arrays, weights):
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.sum()
    stacked = np.column_stack([rank01(a) for a in arrays])
    return stacked @ weights


def power_rank(arrays, p=1.0, weights=None):
    ranks = np.column_stack([rank01(a) for a in arrays])
    if weights is None:
        weights = np.ones(ranks.shape[1])
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.sum()
    if abs(p) < 1e-12:
        # geometric
        return np.exp(np.log(np.clip(ranks, 1e-15, None)) @ weights)
    return (ranks**p @ weights) ** (1.0 / p)


def geo_prob(arrays, weights=None, eps=1e-6):
    mats = np.column_stack([np.clip(a, eps, 1 - eps) for a in arrays])
    if weights is None:
        weights = np.ones(mats.shape[1])
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.sum()
    return np.exp(np.log(mats) @ weights)


def quant_lex(primary, *secondaries, decimals=3):
    keys = [np.round(primary, decimals)] + list(secondaries)
    # lexsort: last key primary
    order = np.lexsort(tuple(reversed(keys)))
    out = np.empty(len(order), dtype=np.float64)
    out[order] = np.arange(1, len(order) + 1)
    return out / (len(order) + 1.0)


# stratified subset for fast weight search (~80k); final AUC always on full OOF
_SUB_IDX = None


def _sub_idx(n=80000):
    global _SUB_IDX
    if _SUB_IDX is None:
        pos = np.where(y == 1)[0]
        neg = np.where(y == 0)[0]
        n_pos = int(n * y.mean())
        n_neg = n - n_pos
        _SUB_IDX = np.concatenate(
            [RNG.choice(pos, n_pos, replace=False), RNG.choice(neg, n_neg, replace=False)]
        )
        RNG.shuffle(_SUB_IDX)
    return _SUB_IDX


def full_oof_auc_rank(keys, weights):
    return roc_auc_score(y, weighted_rank([ext[k]["oof"] for k in keys], weights))


def optimize_rank_weights(keys, n_samples=2500):
    """Maximize OOF AUC via Dirichlet random search on a stratified subset, then score full OOF."""
    k = len(keys)
    idx = _sub_idx()
    y_sub = y[idx]
    ranks = np.column_stack([rank01(ext[name]["oof"][idx]) for name in keys])

    best_w = np.ones(k) / k
    best_a = roc_auc_score(y_sub, ranks @ best_w)
    for _ in range(n_samples):
        w = RNG.dirichlet(np.ones(k) * 1.1)
        a = roc_auc_score(y_sub, ranks @ w)
        if a > best_a:
            best_a, best_w = a, w
    # local refine
    for scale in [0.15, 0.08, 0.04]:
        for _ in range(200):
            noise = RNG.normal(0, scale, size=k)
            w = np.clip(best_w + noise, 1e-4, None)
            w = w / w.sum()
            a = roc_auc_score(y_sub, ranks @ w)
            if a > best_a:
                best_a, best_w = a, w
    full_auc = full_oof_auc_rank(keys, best_w)
    return best_w, full_auc


def apply_rank_weights(te_arrays, weights):
    return weighted_rank(te_arrays, weights)


blend_rows = []
blend_preds = {}


def register(name, pred, oof_proxy=None, note=""):
    path = save_blend(name, pred, note=note)
    row = {"name": name, "path": str(path), "note": note, "mean": float(pred.mean())}
    if oof_proxy is not None:
        row["oof_proxy_auc"] = float(oof_proxy)
        print(f"  oof_proxy_auc={oof_proxy:.6f}")
    blend_rows.append(row)
    blend_preds[name] = pred
    return pred

## 1 — LB-weighted rank of top own runs

Public LB as soft quality prior; rank space so scale differences disappear.

In [4]:
keys_own = ["s94639", "s94637", "s94637b", "s94636", "s94634", "s94629"]
lbs = np.array([own_lb[k] for k in keys_own])
# emphasize the top of the board
w_lb = np.clip(lbs - 0.9460, 1e-6, None) ** 2
w_lb = w_lb / w_lb.sum()
pred01 = weighted_rank([own_te[k] for k in keys_own], w_lb)
register(
    "blend_01_lb_rank_topown",
    pred01,
    note="weights=" + ",".join(f"{k}:{w:.3f}" for k, w in zip(keys_own, w_lb)),
)
print(dict(zip(keys_own, np.round(w_lb, 4))))

saved blend_01_lb_rank_topown.csv              mean=0.500000 std=0.288636 nunique=286,566  weights=s94639:0.201,s94637:0.181,s94637b:0.181,s94636:0.172,s94634:0.153,s94629:0.111
{'s94639': np.float64(0.2014), 's94637': np.float64(0.1813), 's94637b': np.float64(0.1813), 's94636': np.float64(0.1716), 's94634': np.float64(0.1531), 's94629': np.float64(0.1114)}


## 2 — OOF-optimized multi-paradigm rank

Fit rank weights on OOF (`fvE` stands in for our `.94639` anchor). Apply those weights on test with `score94639` replacing `fvE`.

In [5]:
para_keys = ["fvE", "nxgb", "naji", "herm", "ncb", "rtab", "rmlp", "mega"]
w2, auc2 = optimize_rank_weights(para_keys, n_samples=4000)
print("oof auc", auc2)
print(dict(zip(para_keys, np.round(w2, 4))))

# test: swap fvE -> anchor submission
te_arrays = []
for k in para_keys:
    if k == "fvE":
        te_arrays.append(own_te[ANCHOR])
    else:
        te_arrays.append(ext[k]["te"])
pred02 = apply_rank_weights(te_arrays, w2)
register("blend_02_oof_opt_paradigm", pred02, oof_proxy=auc2, note="fvE->s94639 on test")

oof auc 0.9463591620808819
{'fvE': np.float64(0.2834), 'nxgb': np.float64(0.3783), 'naji': np.float64(0.0139), 'herm': np.float64(0.111), 'ncb': np.float64(0.0473), 'rtab': np.float64(0.1659), 'rmlp': np.float64(0.0001), 'mega': np.float64(0.0001)}
saved blend_02_oof_opt_paradigm.csv            mean=0.500000 std=0.288477 nunique=286,571  fvE->s94639 on test
  oof_proxy_auc=0.946359


array([0.57435453, 0.47888942, 0.30083739, ..., 0.07291688, 0.09931635,
       0.00905767], shape=(286571,))

## 3 — Anchor + constrained neural residual

Keep ≥85% of the best own ranking; only a small RealMLP residual (the most diverse high-LB family).

In [6]:
# OOF search: (1-w)*rank(fvE) + w*rank(0.55*rtab+0.45*rmlp), w in [0.04, 0.18]
neural_oof = 0.55 * rank01(ext["rtab"]["oof"]) + 0.45 * rank01(ext["rmlp"]["oof"])
fv_r = rank01(ext["fvE"]["oof"])
best_w, best_a = 0.0, roc_auc_score(y, fv_r)
for w in np.linspace(0.04, 0.18, 29):
    a = roc_auc_score(y, (1 - w) * fv_r + w * neural_oof)
    if a > best_a:
        best_a, best_w = a, float(w)
print(f"residual w={best_w:.3f} oof={best_a:.6f}")

neural_te = 0.55 * rank01(ext["rtab"]["te"]) + 0.45 * rank01(ext["rmlp"]["te"])
pred03 = (1 - best_w) * rank01(own_te[ANCHOR]) + best_w * neural_te
register(
    "blend_03_anchor_neural",
    pred03,
    oof_proxy=best_a,
    note=f"w_neural={best_w:.3f}",
)

residual w=0.180 oof=0.946381
saved blend_03_anchor_neural.csv               mean=0.500000 std=0.288571 nunique=286,484  w_neural=0.180
  oof_proxy_auc=0.946381


array([0.56506907, 0.47875723, 0.30597162, ..., 0.0602989 , 0.10174544,
       0.01030958], shape=(286571,))

## 4 — Power-mean of ranks (p&lt;1)

Down-weights outliers vs arithmetic rank mean; pool = top own + diverse externals.

In [7]:
pow_keys_te = [
    own_te["s94639"],
    own_te["s94637b"],
    own_te["s94636"],
    ext["herm"]["te"],
    ext["nxgb"]["te"],
    ext["ncb"]["te"],
    ext["rtab"]["te"],
    ext["rmlp"]["te"],
    ext["naji"]["te"],
]
# pick p on OOF proxy pool
pow_keys_oof = [
    ext["fvE"]["oof"],
    ext["mega"]["oof"],
    ext["herm"]["oof"],
    ext["nxgb"]["oof"],
    ext["ncb"]["oof"],
    ext["rtab"]["oof"],
    ext["rmlp"]["oof"],
    ext["naji"]["oof"],
    ext["fvD"]["oof"],
]
best_p, best_pa = 1.0, -1.0
for p in [0.35, 0.5, 0.75, 1.0, 1.5, 2.0]:
    a = roc_auc_score(y, power_rank(pow_keys_oof, p=p))
    print(f"p={p:.2f} oof={a:.6f}")
    if a > best_pa:
        best_pa, best_p = a, p
pred04 = power_rank(pow_keys_te, p=best_p)
register("blend_04_power_rank", pred04, oof_proxy=best_pa, note=f"p={best_p}")

p=0.35 oof=0.946377
p=0.50 oof=0.946377
p=0.75 oof=0.946377
p=1.00 oof=0.946376
p=1.50 oof=0.946375
p=2.00 oof=0.946375
saved blend_04_power_rank.csv                  mean=0.499773 std=0.288673 nunique=286,571  p=0.35
  oof_proxy_auc=0.946377


array([0.5705987 , 0.48340261, 0.3015042 , ..., 0.08475622, 0.10341281,
       0.00886575], shape=(286571,))

## 5 — Geometric mean of probabilities

Multiplicative ensemble; good when models agree on extremes and disagree mid-rank.

In [8]:
# convert rank-like own subs (mean~0.5) and raw-prob ext to a common scale via ranks,
# then map back with a geometric mean of *ranks* (stable) AND a true geo of clipped probs
# for raw-prob members. Hybrid: geo of ranks for the selected pool.
geo_te = [
    own_te["s94639"],
    own_te["s94637"],
    ext["nxgb"]["te"],
    ext["herm"]["te"],
    ext["ncb"]["te"],
    ext["rtab"]["te"],
    ext["rmlp"]["te"],
]
geo_oof = [
    ext["fvE"]["oof"],
    ext["mega"]["oof"],
    ext["nxgb"]["oof"],
    ext["herm"]["oof"],
    ext["ncb"]["oof"],
    ext["rtab"]["oof"],
    ext["rmlp"]["oof"],
]
# LB-ish weights
geo_w = np.array([0.94639, 0.94637, 0.94639, 0.94627, 0.94624, 0.94620, 0.94621])
geo_w = np.clip(geo_w - 0.9460, 1e-4, None)
geo_w = geo_w / geo_w.sum()
auc5 = roc_auc_score(y, power_rank(geo_oof, p=0.0, weights=geo_w))  # geometric ranks
pred05 = power_rank(geo_te, p=0.0, weights=geo_w)
register("blend_05_geo_rank", pred05, oof_proxy=auc5, note="geometric mean of ranks")

saved blend_05_geo_rank.csv                    mean=0.499643 std=0.288805 nunique=286,571  geometric mean of ranks
  oof_proxy_auc=0.946375


array([0.56957354, 0.48186986, 0.30248354, ..., 0.07168678, 0.10276866,
       0.00910381], shape=(286571,))

## 6 — Caruana greedy ensemble (OOF)

Forward-select members that improve OOF AUC; map `fvE`→`s94639` on test.

In [9]:
cand_keys = ["fvE", "mega", "naji", "nxgb", "herm", "ncb", "rtab", "rmlp", "fvD", "pub"]
idx = _sub_idx()
y_sub = y[idx]
# seed with best single on full OOF
best_single = max(cand_keys, key=lambda k: roc_auc_score(y, ext[k]["oof"]))
selected = [best_single]
cur_auc = roc_auc_score(y_sub, rank01(ext[best_single]["oof"][idx]))
print(f"seed {best_single} sub_auc={cur_auc:.6f}")
for step in range(6):
    best = None
    for k in cand_keys:
        if k in selected:
            continue
        trial = selected + [k]
        # try a few random weightings on subset (not just equal)
        local_best = -1.0
        local_w = np.ones(len(trial)) / len(trial)
        for wtry in [local_w] + [RNG.dirichlet(np.ones(len(trial)) * 1.3) for _ in range(40)]:
            a = roc_auc_score(
                y_sub, weighted_rank([ext[x]["oof"][idx] for x in trial], wtry)
            )
            if a > local_best:
                local_best, local_w = a, wtry
        if local_best > cur_auc + 1e-7:
            best = (local_best, k, local_w, trial)
    if best is None:
        break
    cur_auc, k_add, w6, selected = best[0], best[1], best[2], best[3]
    print(f"+{k_add:5s} sub_auc={cur_auc:.6f} n={len(selected)}")

w6, auc6 = optimize_rank_weights(selected, n_samples=5000)
print("final", selected, "auc", auc6, "w", np.round(w6, 4))
te6 = []
for k in selected:
    te6.append(own_te[ANCHOR] if k == "fvE" else ext[k]["te"])
pred06 = apply_rank_weights(te6, w6)
register("blend_06_caruana_oof", pred06, oof_proxy=auc6, note="sel=" + "+".join(selected))

seed fvE sub_auc=0.945541
+pub   sub_auc=0.945568 n=2
+rmlp  sub_auc=0.945583 n=3
+rtab  sub_auc=0.945593 n=4
+ncb   sub_auc=0.945615 n=5
+fvD   sub_auc=0.945621 n=6
+herm  sub_auc=0.945628 n=7
final ['fvE', 'pub', 'rmlp', 'rtab', 'ncb', 'fvD', 'herm'] auc 0.9463809900169754 w [3.989e-01 1.000e-04 1.000e-04 1.904e-01 2.656e-01 1.000e-04 1.447e-01]
saved blend_06_caruana_oof.csv                 mean=0.500000 std=0.288441 nunique=286,571  sel=fvE+pub+rmlp+rtab+ncb+fvD+herm
  oof_proxy_auc=0.946381


array([0.57099018, 0.48487388, 0.30208686, ..., 0.08189246, 0.10709323,
       0.00877947], shape=(286571,))

## 7 — Quantized lexsort tie-break

Round the anchor to create ties, break with neural then XGB/herm (OOF-picked decimals).

In [10]:
best7 = (-1.0, None)
for dec in [2, 3, 4]:
    for s1, s2 in [("rtab", "herm"), ("rmlp", "herm"), ("rtab", "nxgb"), ("ncb", "herm"), ("rtab", "rmlp")]:
        oo = quant_lex(ext["fvE"]["oof"], ext[s1]["oof"], ext[s2]["oof"], decimals=dec)
        a = roc_auc_score(y, oo)
        if a > best7[0]:
            best7 = (a, (dec, s1, s2))
print("best qlex", best7)
dec, s1, s2 = best7[1]
pred07 = quant_lex(own_te[ANCHOR], ext[s1]["te"], ext[s2]["te"], decimals=dec)
register(
    "blend_07_qlex_tiebreak",
    pred07,
    oof_proxy=best7[0],
    note=f"dec={dec} {s1}->{s2}",
)

best qlex (0.946360204941405, (2, 'rmlp', 'herm'))
saved blend_07_qlex_tiebreak.csv               mean=0.500000 std=0.288674 nunique=286,571  dec=2 rmlp->herm
  oof_proxy_auc=0.946360


array([0.56566587, 0.48290482, 0.30578703, ..., 0.05413299, 0.10324805,
       0.01179459], shape=(286571,))

## 8 — Wide OOF stack + blend-of-blends fuse

(a) Dirichlet search for rank-weights on a wide paradigm pool.  
(b) Rank-fuse that stack with blends 02/03/06 (strongest OOF-guided ones).

In [11]:
de_keys = ["fvE", "nxgb", "naji", "herm", "ncb", "rtab", "rmlp", "mega", "fvD", "pub"]
w8a, auc8a = optimize_rank_weights(de_keys, n_samples=6000)
print("DE-style search oof", auc8a)
print(dict(zip(de_keys, np.round(w8a, 4))))

te8a = []
for k in de_keys:
    te8a.append(own_te[ANCHOR] if k == "fvE" else ext[k]["te"])
pred_de = apply_rank_weights(te8a, w8a)

# fuse with other strong blends in rank space; weights from proxy OOF when available
fuse_names = [
    "blend_02_oof_opt_paradigm",
    "blend_03_anchor_neural",
    "blend_06_caruana_oof",
]
fuse_w = []
fuse_te = [pred_de]
# DE gets weight from its oof
fuse_w.append(max(auc8a - 0.9460, 1e-4))
# for other blends, use registered oof_proxy
proxy = {r["name"]: r.get("oof_proxy_auc", 0.9463) for r in blend_rows}
for nm in fuse_names:
    fuse_te.append(blend_preds[nm])
    fuse_w.append(max(proxy[nm] - 0.9460, 1e-4))
fuse_w = np.asarray(fuse_w, dtype=np.float64)
fuse_w = fuse_w / fuse_w.sum()
pred08 = weighted_rank(fuse_te, fuse_w)
# proxy: weighted avg of component proxy aucs (DE + listed)
auc8 = float(fuse_w[0] * auc8a + np.dot(fuse_w[1:], [proxy[n] for n in fuse_names]))
register(
    "blend_08_de_fuse",
    pred08,
    oof_proxy=auc8a,
    note="DE+" + "+".join(fuse_names) + f" fuse_w={np.round(fuse_w,3)}",
)

DE-style search oof 0.9463530012579389
{'fvE': np.float64(0.1689), 'nxgb': np.float64(0.4068), 'naji': np.float64(0.0001), 'herm': np.float64(0.1516), 'ncb': np.float64(0.0494), 'rtab': np.float64(0.1854), 'rmlp': np.float64(0.0001), 'mega': np.float64(0.0001), 'fvD': np.float64(0.0374), 'pub': np.float64(0.0001)}
saved blend_08_de_fuse.csv                     mean=0.500000 std=0.288658 nunique=286,571  DE+blend_02_oof_opt_paradigm+blend_03_anchor_neural+blend_06_caruana_oof fuse_w=[0.239 0.244 0.259 0.258]
  oof_proxy_auc=0.946353


array([0.57102147, 0.48101701, 0.30172653, ..., 0.07186587, 0.10274095,
       0.00818338], shape=(286571,))

## Summary

In [12]:
assert len(blend_rows) == 8, len(blend_rows)
summary = pd.DataFrame(blend_rows)
summary_path = OUT_DIR / "summary.csv"
summary.to_csv(summary_path, index=False)
print(summary[["name", "oof_proxy_auc", "mean", "note"]].to_string(index=False))
print()
print(f"wrote {len(blend_rows)} files to {OUT_DIR.resolve()}")

# pairwise spearman among the 8 (want some spread)
names = [r["name"] for r in blend_rows]
print("\nSpearman among blends:")
for i, a in enumerate(names):
    for b in names[i + 1 :]:
        sp = spearmanr(blend_preds[a], blend_preds[b]).correlation
        print(f"  {a.replace('blend_','')} vs {b.replace('blend_','')}: {sp:.6f}")

# vs anchor
print("\nSpearman vs anchor s94639:")
for a in names:
    print(f"  {a}: {spearmanr(blend_preds[a], own_te[ANCHOR]).correlation:.6f}")

                     name  oof_proxy_auc     mean                                                                                                      note
  blend_01_lb_rank_topown            NaN 0.500000                    weights=s94639:0.201,s94637:0.181,s94637b:0.181,s94636:0.172,s94634:0.153,s94629:0.111
blend_02_oof_opt_paradigm       0.946359 0.500000                                                                                       fvE->s94639 on test
   blend_03_anchor_neural       0.946381 0.500000                                                                                            w_neural=0.180
      blend_04_power_rank       0.946377 0.499773                                                                                                    p=0.35
        blend_05_geo_rank       0.946375 0.499643                                                                                   geometric mean of ranks
     blend_06_caruana_oof       0.946381 0.500000               